# Counterfeit Pharmaceutical Packaging Detection: run on Google Colab

Runs the whole project on a free Colab GPU: data cleaning, splits, SVM baseline, CNN experiments, tables and figures, Grad-CAM and the demo.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`, then run the cells from top to bottom.

## 1. Get the code

In [ ]:
REPO = 'https://github.com/abhinavvt9/counterfeit-packaging-cnn.git'
import os
if not os.path.exists('/content/counterfeit-packaging-cnn'):
    !git clone -q {REPO} /content/counterfeit-packaging-cnn
os.chdir('/content/counterfeit-packaging-cnn')
!ls

## 2. Get the dataset
Download `archive.zip` from Kaggle: https://www.kaggle.com/datasets/surajkumarjha1/fake-vs-real-medicine-datasets-images

Then run this cell and upload the zip when asked.

In [ ]:
import zipfile
if not os.path.isdir('dataset'):
    from google.colab import files
    up = files.upload()
    for name in up:
        zipfile.ZipFile(name).extractall('dataset')
!ls dataset

## 3. Install libraries (PyTorch is already on Colab)

In [ ]:
!pip -q install gradio scikit-image opencv-python-headless
import torch; print('PyTorch', torch.__version__, '| GPU:', torch.cuda.is_available())

## 4. Data stages and SVM baseline (CPU, about 5-10 minutes)
Cleaning (MD5 + perceptual hash), shortcut audit, grouped 5-fold splits with leakage check, preprocessing, and the SVM baseline.

In [ ]:
!python run_all.py --data dataset --stages index audit splits cache svm

## 5. Smoke test (about 1 minute)
Must end with **SMOKE TEST PASSED**.

In [ ]:
!python smoke_test.py

## 6. CNN experiments, tables, Grad-CAM and demo model
The CNN results of the paper are already stored in `results/runs/`, so finished folds are skipped. To retrain all 7 experiments from scratch (about 30-60 minutes on a T4), run the first line too.

In [ ]:
# !rm -f results/runs/E*.json          # uncomment to retrain every CNN from scratch
!python run_all.py --data dataset --stages cnn report errors final
!python analysis/significance.py

## 7. Results

In [ ]:
import pandas as pd
from IPython.display import Image, display
for t in ['table_svm_vs_cnn', 'table_cnn_experiments', 'table_shortcut_study']:
    print('\n==', t); display(pd.read_csv(f'results/tables/{t}.csv'))
for f in ['figures/confusion_matrices.png', 'figures/roc_curves.png', 'figures/cnn_experiments.png',
          'figures/training_curves.png', 'figures/shortcut_study.png', 'error_analysis/gradcam_examples.png']:
    print(f); display(Image('results/' + f, width=850))

## 8. Demo
Prints a public link. Open it, upload a packaging photo and press **Analyze**. Stop the cell to end the demo.

In [ ]:
!python demo/app.py